# handwave in Jupyter

A webcam widget that tracks your hands. Run the cells in order, and allow camera access when your browser asks.

*For widget development only:* reload Python changes, and hot-reload the widget's JavaScript and CSS (run `npm run dev` in another terminal).

In [ ]:
# %load_ext autoreload
# %autoreload 2
# %env ANYWIDGET_HMR=1

In [ ]:
from handwave import HandwaveWidget

w = HandwaveWidget(max_num_hands=2, width=320, height=240)
w

## Landmarks

`w.hand` is the first hand's 21 landmarks as a data frame (`w.hands` has them all), as of when you run the cell. `x` and `y` run from 0 to 1 across the camera frame. Hold a hand up and run this:

In [ ]:
w.hand

In [ ]:
w.hand.attrs if w.hand is not None else "No hands in view"

## Gestures

Gestures call back into Python as they happen. Here they drive some ordinary ipywidgets:

- **Pinch** (make an "OK" sign, hold it still for a moment) and twist your hand like a dial to turn the dial.
- **Clap twice** (clap, brief pause, clap) to reset everything.
- **Reel** to step: point double-barrelled finger guns (index and middle fingers out) at each other, hold your left hand still, and circle your right hand: clockwise (as you see it in the preview) to step forward, the other way to step back.

The last gesture event shows underneath, including fists and finger guns.

In [ ]:
import math

import ipywidgets as widgets

dial = widgets.FloatSlider(0, min=-180, max=180, description="Dial (°)", readout_format=".0f")
step = widgets.IntSlider(0, min=0, max=10, description="Step")
claps = widgets.IntText(0, description="Claps", disabled=True)
last = widgets.Label()


# (Running this cell again registers the callbacks again; run the widget cell first.)
@w.on_clutch
def turn_dial(event):
    last.value = f"{event.hand} {event.gesture}: {event.type}"
    if event.gesture == "pinch" and event.type == "move":
        dial.value = min(dial.max, max(dial.min, dial.value + math.degrees(event.dtheta)))


@w.on_clap
def reset(event):
    claps.value += 1
    dial.value = 0
    step.value = 0


@w.on_reel
def step_through(event):
    step.value = min(step.max, max(step.min, step.value + event.direction))


widgets.VBox([dial, step, claps, last])